In [1]:
model_info =  {
    "Transformer_Layer" : 6,
    "batch_size" : 256,
    "Attention_Heads" : 8,
    "Embedding_Dimension" : 512,
    "Max Sequence Length" : 128,
    "Dropout" : 0.1,
    "Vocab_size" : 50257,
    "Number_of_Output" : 3
}

In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import os
import math
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from torch.utils.data import DataLoader

In [3]:
splits = {'train': 'train.json', 'validation': 'validation.json', 'test': 'test.json'}
df = pd.read_json("hf://datasets/bdstar/twitter-sentiment-analysis/" + splits["train"])

In [4]:
label_mapping = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}
df['label_encoded'] = df['label'].map(label_mapping)
df.head(),df.value_counts('label')

(   ID                                               text     label  \
 0   1  is upset that he can't update his Facebook by ...  negative   
 1   2  @Kenichan I dived many times for the ball. Man...  negative   
 2   3     my whole body feels itchy and like its on fire  negative   
 3   4  @nationwideclass no, it's not behaving at all....  negative   
 4   5                       @Kwesidei not the whole crew  negative   
 
    label_encoded  
 0              0  
 1              0  
 2              0  
 3              0  
 4              0  ,
 label
 negative    1570031
 positive    1561453
 neutral       10725
 Name: count, dtype: int64)

In [5]:
!pip install transformers datasets accelerate

First, let's split our data into training and validation sets. Then, we'll initialize a tokenizer that uses Byte Pair Encoding (BPE), such as the one from the `gpt2` model.

In [6]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label_encoded'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

Next, we'll define a tokenization function to process the 'text' column and apply it to both the training and validation datasets. We'll also set up a `DataCollatorWithPadding` to handle dynamic padding during batching, which is crucial for variable-length sequences.

In [7]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT2 has no PAD token by default
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

def tokenize_function(examples):
    # Ensure tokenization and padding adhere to the model's max sequence length
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=model_info["Max Sequence Length"], # Use the model's defined max length
        padding='max_length' # Pad to this max_length, not just to the longest in batch
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

tokenized_train_dataset = tokenized_train_dataset.remove_columns(['text', 'ID', 'label'])
tokenized_val_dataset = tokenized_val_dataset.remove_columns(['text', 'ID', 'label'])

tokenized_train_dataset = tokenized_train_dataset.rename_column('label_encoded', 'labels')
tokenized_val_dataset = tokenized_val_dataset.rename_column('label_encoded', 'labels')

tokenized_train_dataset.set_format('torch')
tokenized_val_dataset.set_format('torch')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/2827988 [00:00<?, ? examples/s]

Map:   0%|          | 0/314221 [00:00<?, ? examples/s]

Finally, we'll create the PyTorch `DataLoader` objects for both the training and validation sets. These DataLoaders will feed batches of tokenized and padded data to your Transformer model during training.

In [8]:
batch_size = 128
train_dataloader = DataLoader(
    tokenized_train_dataset,
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator
)
val_dataloader = DataLoader(
    tokenized_val_dataset,
    batch_size=batch_size,
    collate_fn=data_collator
)

print(f"Training DataLoader has {len(train_dataloader)} batches.")
print(f"Validation DataLoader has {len(val_dataloader)} batches.")

for batch in train_dataloader:
    print(f"Input IDs shape: {batch['input_ids'].shape}")
    print(f"Attention Mask shape: {batch['attention_mask'].shape}")
    print(f"Labels shape: {batch['labels'].shape}")
    break

Training DataLoader has 22094 batches.
Validation DataLoader has 2455 batches.
Input IDs shape: torch.Size([128, 128])
Attention Mask shape: torch.Size([128, 128])
Labels shape: torch.Size([128])


In [9]:
import torch
import torch.nn as nn

class MultiLatentAttention(nn.Module) :
    def __init__(self , d_in,d_out , context_length ,dropout ,num_heads , qkv_bias = False ):
        super().__init__()

        assert (d_out % num_heads == 0)
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x, attention_mask=None): # Added attention_mask parameter
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)


        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)

        # Apply padding mask if provided
        if attention_mask is not None:
            # Expand attention_mask to match the dimensions of attn_scores for masking keys (last dim)
            padding_mask_keys = (attention_mask == 0).unsqueeze(1).unsqueeze(2) # (B, 1, 1, S)
            attn_scores.masked_fill_(padding_mask_keys, -torch.inf)

            # Expand attention_mask to match the dimensions of attn_scores for masking queries (second to last dim)
            padding_mask_queries = (attention_mask == 0).unsqueeze(1).unsqueeze(3) # (B, 1, S, 1)
            attn_scores.masked_fill_(padding_mask_queries, -torch.inf)

        # Apply causal mask (which is independent of batch or heads)
        causal_mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # Expand causal mask to match attn_scores dimensions (batch, heads, seq_len, seq_len)
        causal_mask_expanded = causal_mask_bool[None, None, :, :].expand_as(attn_scores)
        attn_scores.masked_fill_(causal_mask_expanded, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

In [10]:
class GELU(nn.Module) :
    def __init__(self) :
        super().__init__()

    def forward(self , x) :
        return 0.5 * x * (1+torch.tanh(
            torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x,3))
        ))

In [11]:
class FeedForward(nn.Module) :
    def __init__(self,dim,dropout) :
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(dim,4*dim),
            GELU(),
            nn.Linear(4*dim,dim),
            nn.Dropout(dropout)
        )

    def forward(self,x) :
        return self.layers(x)

In [12]:
class LayerNorm(nn.Module) :
    def __init__(self, emb_dim) :
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self , x):
        mean = x.mean(dim=-1,keepdim = True)
        var = x.var(dim=-1,keepdim = True , unbiased=False )
        nor_out = (x - mean) / torch.sqrt(var + self.eps)
        return nor_out * self.scale + self.shift


In [13]:
class Transformer(nn.Module) :
    def __init__(self,model_info) :
        super().__init__()
        self.attn = MultiLatentAttention(
            d_in=model_info["Embedding_Dimension"],
            d_out=model_info["Embedding_Dimension"],
            context_length=model_info["Max Sequence Length"],
            dropout=model_info["Dropout"],
            num_heads=model_info["Attention_Heads"]
        )
        self.ff = FeedForward(model_info["Embedding_Dimension"],model_info["Dropout"])
        self.ln1 = LayerNorm(model_info["Embedding_Dimension"])
        self.ln2 = LayerNorm(model_info["Embedding_Dimension"])
        self.dropout = nn.Dropout(model_info["Dropout"])

    def forward(self, x, attention_mask=None) :
        shortcut = x
        x =  self.ln1(x)
        x = self.attn(x, attention_mask) # Pass attention_mask to MultiLatentAttention
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.ln2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

In [14]:
class TirZ_M1_Mini_Encoder(nn.Module) :
    def __init__(self,model_info) :
        super().__init__()
        self.token_emd = nn.Embedding(model_info["Vocab_size"],model_info["Embedding_Dimension"])
        self.pos_emd = nn.Embedding(model_info["Max Sequence Length"],model_info["Embedding_Dimension"])
        self.dropout = nn.Dropout(model_info["Dropout"])
        self.transformerBlocks = nn.ModuleList( # Use ModuleList for proper registration
            [Transformer(model_info) for _ in range(model_info["Transformer_Layer"])]
        )
        self.norm = LayerNorm(model_info["Embedding_Dimension"])

        self.final_layer = nn.Linear(model_info["Embedding_Dimension"] , model_info["Number_of_Output"])

    def forward(self, x, attention_mask=None) : # Added attention_mask parameter
        batch_size , seq_length = x.shape
        token_emb = self.token_emd(x)
        pos_emb = self.pos_emd(torch.arange(seq_length).to(x.device))
        x = token_emb + pos_emb

        x = self.dropout(x)
        for block in self.transformerBlocks:
            x = block(x, attention_mask) # Pass attention_mask to each Transformer block

        x = self.norm(x)
        # For sequence-level classification, we need to aggregate the sequence dimension.
        # Taking the mean across the sequence is a common pooling strategy.
        x = x.mean(dim=1) # Shape changes from (batch_size, seq_length, emb_dim) to (batch_size, emb_dim)

        x = self.final_layer(x)
        return x

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [16]:
model = TirZ_M1_Mini_Encoder(model_info)
model.to(device)

TirZ_M1_Mini_Encoder(
  (token_emd): Embedding(50257, 512)
  (pos_emd): Embedding(128, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformerBlocks): ModuleList(
    (0-5): 6 x Transformer(
      (attn): MultiLatentAttention(
        (W_query): Linear(in_features=512, out_features=512, bias=False)
        (W_key): Linear(in_features=512, out_features=512, bias=False)
        (W_value): Linear(in_features=512, out_features=512, bias=False)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
      (ln1): LayerNorm()
      (ln2): LayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm()
  (final_lay

In [17]:

def plot_losses(train_losses, val_losses):
    """Plots the training and validation losses over epochs."""
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

def train_model(model, train_dataloader, val_dataloader, epochs, optimizer, criterion, device):
    """Trains the given model and plots the training and validation losses."""
    model.to(device)
    train_losses = []
    val_losses = []
    scaler = torch.cuda.amp.GradScaler() # Initialize GradScaler for mixed precision

    for epoch in range(epochs):
        model.train() # Set model to training mode
        total_train_loss = 0
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device).long() # Ensure labels are long type

            optimizer.zero_grad()

            # Use autocast for mixed precision training
            with torch.cuda.amp.autocast():
                outputs = model(input_ids, attention_mask) # Pass attention_mask to the model
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward() # Scale loss and perform backward pass
            scaler.step(optimizer) # Update optimizer with scaled gradients
            scaler.update() # Update the scale for the next iteration

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_dataloader)
        train_losses.append(avg_train_loss)

        model.eval() # Set model to evaluation mode
        total_val_loss = 0
        with torch.no_grad(): # Disable gradient calculations for validation
            for batch in tqdm(val_dataloader, desc=f"Epoch {epoch+1} Validation"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device).long() # Ensure labels are long type

                with torch.cuda.amp.autocast(): # Use autocast for mixed precision inference
                    outputs = model(input_ids, attention_mask) # Pass attention_mask to the model
                    loss = criterion(outputs, labels)

                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")

    print("\nTraining complete!")
    plot_losses(train_losses, val_losses)
    return model

In [ ]:
epochs = 3
learning_rate = 5e-5

# Re-instantiate the model to ensure it uses the latest class definitions
model = TirZ_M1_Mini_Encoder(model_info)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Call the train_model function
trained_model = train_model(model, train_dataloader, val_dataloader, epochs, optimizer, criterion, device)

/tmp/ipykernel_8229/3671543061.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # Initialize GradScaler for mixed precision
Epoch 1 Training:   0%|          | 0/22094 [00:00<?, ?it/s]/tmp/ipykernel_8229/3671543061.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1 Training:   3%|▎         | 667/22094 [04:50<2:35:47,  2.29it/s]